# Local code knowledge graph

This walkthrough builds a typed graph from a local Git repository, records reproducible repository provenance, retrieves relevant code with TF-IDF and graph expansion, and renders interactive Plotly views. The implementation lives in the `code_knowledge_graph` package so the notebook contains only configuration and analysis.

## Configure the target

Pass a repository path directly for automation. In an interactive session, `GraphConfig.resolve` checks `CODE_GRAPH_TARGET` and then an `implicit-decision-gate` directory beside this notebook repository. Use `mode="public"` or `mode="demo"` with a full `expected_commit` when publishing an artifact.

In [ ]:
from pathlib import Path

from code_knowledge_graph import (
    GraphConfig,
    build_knowledge_graph,
    build_search_index,
    graph_summary,
    query_graph,
    query_result_figure,
    repository_overview_figure,
    write_graph_artifact,
    write_plotly_html,
)

notebook_path = (Path.cwd() / "knowledge_code_graph.ipynb").resolve()
config = GraphConfig.resolve(notebook_path=notebook_path)
config.repository_root.name

## Build and inspect the graph

Python files produce file, class, method, function, fixture, and test nodes. Typed edges capture containment, imports, inheritance, instantiation, calls, tests, and Git co-change evidence. Other recognized source files remain searchable file nodes.

In [ ]:
knowledge = build_knowledge_graph(config)
node_summary, edge_summary = graph_summary(knowledge)
knowledge.snapshot

In [ ]:
node_summary, edge_summary

## Explore repository relationships

The overview aggregates symbol relationships at file level. Plotly's MIME renderer keeps the saved notebook portable and interactive without an iframe or an absolute artifact URL.

In [ ]:
overview_figure = repository_overview_figure(knowledge)
overview_figure.show(
    renderer="plotly_mimetype",
    config={"displaylogo": False, "responsive": True, "scrollZoom": True},
)

Interactive Plotly overview. Run the cell to refresh it for the configured repository.

## Retrieve direct and related code

The search index combines word and identifier character TF-IDF features. Query expansion follows the strongest weighted paths from the direct anchors. Every related row includes its path so the result remains inspectable.

In [ ]:
search_index = build_search_index(knowledge.nodes)
query = "where does a run pause for owner decisions and then resume?"
result = query_graph(knowledge, search_index, query)
result.relevant, result.related

In [ ]:
query_figure = query_result_figure(knowledge, result)
query_figure.show(
    renderer="plotly_mimetype",
    config={"displaylogo": False, "responsive": True, "scrollZoom": True},
)

Interactive Plotly query graph. Run the cell to refresh it for the current query.

## Save deterministic artifacts

The JSON artifact is canonical and excludes the absolute repository root. The standalone HTML files include Plotly for offline viewing. Run this optional cell when you want files under the local `artifacts` directory.

In [ ]:
artifact_directory = Path.cwd() / "artifacts"
write_graph_artifact(knowledge, artifact_directory / "knowledge_graph.json")
write_plotly_html(overview_figure, artifact_directory / "repository_overview.html")
write_plotly_html(query_figure, artifact_directory / "query_graph.html");